In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import mne
import matplotlib.pyplot as plt
import matplotlib as mpl

In [ ]:
SFREQ = 256
BANDPASS = (1, 20)
TMIN, TMAX = -0.3, 1.0
BASELINE = (None, 0)
PEAK_TO_PEAK = dict(eeg=400e-6)

ELECTRODES = ["AF7", "AF8", "TP9", "TP10"]

EEG_DIR = Path("muse")
BEH_DIR = Path("data")

CONDITION_LABELS = {
    "silence": "Silence",
    "lyrics": "Avec paroles",
    "no_lyrics": "Sans paroles",
}
CONDITION_COLORS = {
    "silence": "#4472C4",
    "lyrics": "#ED7D31",
    "no_lyrics": "#70AD47",
}

TEXT_COLOR = "#595959"
mpl.rcParams.update({
    "font.family": "Calibri",
    "text.color": TEXT_COLOR,
    "axes.labelcolor": TEXT_COLOR,
    "xtick.color": TEXT_COLOR,
    "ytick.color": TEXT_COLOR,
    "axes.edgecolor": "#D9D9D9",
    "axes.grid": True,
    "grid.color": "#D9D9D9",
    "grid.linewidth": 0.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "figure.dpi": 150,
})

In [ ]:
class ERPRecording:
    def __init__(
        self,
        eeg_df: pd.DataFrame,
        behav_df: pd.DataFrame,
        tmin: float = TMIN,
        tmax: float = TMAX,
        sfreq: int = SFREQ,
        baseline: tuple = BASELINE,
    ) -> None:
        if eeg_df.empty or behav_df.empty:
            raise ValueError("DataFrames must not be empty.")

        self.sfreq = sfreq
        self.tmin = tmin
        self.tmax = tmax
        self.grand_avg_dict: dict = {}
        self.n_epochs: dict = {}

        if "trial_type" in behav_df.columns:
            behav_df = behav_df[behav_df["trial_type"] == "nogo"]

        stim_labels = behav_df["condition"].unique()
        self.event_dict = {label: i + 1 for i, label in enumerate(stim_labels)}

        eeg_ts = eeg_df["timestamps"].values
        marker_codes = behav_df["condition"].map(self.event_dict).values
        marker_ts = behav_df["marker_ts"].values

        ins = np.searchsorted(eeg_ts, marker_ts)
        ins = np.clip(ins, 1, len(eeg_ts) - 1)
        left = np.abs(marker_ts - eeg_ts[ins - 1])
        right = np.abs(marker_ts - eeg_ts[ins])
        nearest = np.where(left <= right, ins - 1, ins)

        marker_series = pd.Series(0, index=eeg_df.index)
        marker_series.iloc[nearest] = marker_codes

        drop = ["timestamps"]
        if "Right AUX" in eeg_df.columns:
            drop.append("Right AUX")
        channels = [c for c in eeg_df.columns if c not in drop]
        eeg_data = eeg_df[channels].apply(pd.to_numeric, errors="coerce").fillna(0).T.values * 1e-6
        eeg_data = np.vstack([eeg_data, marker_series.values[np.newaxis, :]])

        info = mne.create_info(
            ch_names=channels + ["markers"],
            sfreq=sfreq,
            ch_types=["eeg"] * len(channels) + ["stim"],
        )
        raw = mne.io.RawArray(eeg_data, info)
        raw.filter(*BANDPASS)
        raw.set_eeg_reference("average", projection=False)

        events = mne.find_events(raw)
        n_total = len(events)
        epochs = mne.Epochs(
            raw, events, event_id=self.event_dict,
            tmin=tmin, tmax=tmax, baseline=baseline,
            reject=PEAK_TO_PEAK, preload=True,
        )
        n_accepted = len(epochs)

        for label in stim_labels:
            try:
                ep = epochs[label]
                self.n_epochs[label] = len(ep)
                if len(ep) > 0:
                    self.grand_avg_dict[label] = ep.average()
            except Exception:
                self.n_epochs[label] = 0

        print(f"   Epochs: {n_accepted}/{n_total}")


class GrandAverageERP:
    def __init__(self, recordings: list) -> None:
        if not recordings:
            raise ValueError("No recordings provided.")

        self.n = len(recordings)
        print(f"Grand average from {self.n} participant(s):")

        all_labels: set = set()
        for rec in recordings:
            all_labels.update(rec.grand_avg_dict.keys())

        self.grand_avg_dict: dict = {}
        self.n_per_condition: dict = {}

        for label in sorted(all_labels):
            evokeds = [
                rec.grand_avg_dict[label]
                for rec in recordings
                if label in rec.grand_avg_dict
            ]
            self.n_per_condition[label] = len(evokeds)
            if len(evokeds) == 1:
                self.grand_avg_dict[label] = evokeds[0]
            elif len(evokeds) > 1:
                self.grand_avg_dict[label] = mne.grand_average(evokeds)
            print(f"  {label}: {len(evokeds)} participant(s)")

    def plot(self, electrode: str, title: str,
             components: list | None = None,
             legend_loc: str = "upper left") -> None:
        """Plot grand-average ERPs at a single electrode."""
        fig, ax = plt.subplots(figsize=(7, 4))
        for label in ["silence", "lyrics", "no_lyrics"]:
            if label not in self.grand_avg_dict:
                continue
            evoked = self.grand_avg_dict[label]
            data = evoked.get_data(picks=[electrode])[0] * 1e6
            display_name = CONDITION_LABELS.get(label, label)
            color = CONDITION_COLORS.get(label, None)
            ax.plot(evoked.times, data, label=display_name,
                    color=color, linewidth=1.5)
        ax.invert_yaxis()
        ax.set_xlabel("Temps (s)", fontsize=10)
        ax.set_ylabel("Amplitude (\u00b5V)", fontsize=10)
        ax.set_title(title, fontsize=14)
        ax.tick_params(labelsize=9)
        ax.legend(fontsize=9, frameon=False, loc=legend_loc)
        if components:
            top = min(ax.get_ylim())
            span = abs(ax.get_ylim()[0] - ax.get_ylim()[1])
            for comp in components:
                t0, t1 = comp["window"]
                c = comp.get("color", "#888888")
                ax.axvspan(t0, t1, color=c, alpha=0.13, linewidth=0)
                ax.text((t0 + t1) / 2, top + span * 0.05,
                        comp["label"], ha="center", fontsize=9,
                        fontstyle="italic", color=c)
        plt.tight_layout()
        plt.show()

In [ ]:
participant_files = []
for eeg_path in sorted(EEG_DIR.glob("*.csv")):
    beh_path = BEH_DIR / eeg_path.name
    if beh_path.exists():
        participant_files.append((str(eeg_path), str(beh_path)))

N_PARTICIPANTS = len(participant_files)
print(f"Found {N_PARTICIPANTS} participant(s).")

In [ ]:
def load_and_clean(eeg_path: str, beh_path: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    eeg_df = pd.read_csv(eeg_path, on_bad_lines="skip")
    beh_raw = pd.read_csv(beh_path)

    eeg_df["timestamps"] = pd.to_numeric(eeg_df["timestamps"], errors="coerce")
    beh_raw["stimulus_unix"] = pd.to_numeric(beh_raw["stimulus_unix"], errors="coerce")

    eeg_df = eeg_df[eeg_df["timestamps"].notnull()]
    beh_raw = beh_raw[beh_raw["stimulus_unix"].notnull()]

    eeg_df["timestamps"] = pd.to_datetime(eeg_df["timestamps"], unit="s")
    beh_raw["marker_ts"] = pd.to_datetime(beh_raw["stimulus_unix"], unit="s")

    eeg_start, eeg_end = eeg_df["timestamps"].iloc[0], eeg_df["timestamps"].iloc[-1]
    beh_raw = beh_raw[(beh_raw["marker_ts"] >= eeg_start) & (beh_raw["marker_ts"] <= eeg_end)]

    return eeg_df, beh_raw


sessions: list = []

for i, (eeg_path, beh_path) in enumerate(participant_files, start=1):
    print(f"\nParticipant {i}/{N_PARTICIPANTS}")
    try:
        eeg_df, beh_raw = load_and_clean(eeg_path, beh_path)
        rec = ERPRecording(eeg_df, beh_raw)
        sessions.append(rec)
    except Exception as e:
        print(f"   Skipped: {e}")

print(f"\nProcessed {len(sessions)}/{N_PARTICIPANTS} participant(s).")
grand = GrandAverageERP(sessions)

In [ ]:
n = max(grand.n_per_condition.values())

ERP_COMPONENTS = [
    {"label": "N2",  "window": (0.20, 0.30), "color": "#E8751A"},
    {"label": "P3",  "window": (0.30, 0.60), "color": "#4A90D9"},
]

LEGEND_LOC = {
    "TP9":  "upper left",
    "TP10": "upper left",
}

for fig_num, electrode in enumerate(ELECTRODES, start=8):
    grand.plot(
        electrode=electrode,
        title=f"Figure {fig_num}. Potentiel évoqué moyen ({electrode})",
        components=ERP_COMPONENTS,
        legend_loc=LEGEND_LOC.get(electrode, "lower left"),
    )